# Ejercicios — Clase 6: LOB Data Science Pipeline

| Tier | Ejercicios | Cuándo |
|------|-----------|--------|
| **Núcleo** | 1–5 | Obligatorio en clase |
| **Si vamos bien** | 6–7 | Si el ritmo lo permite |
| **Bonus / casa** | 8–10 | Tarea o alumnos adelantados |

Los datos están en `../data/lob_features.csv` (generado por `lesson.ipynb`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

plt.style.use('dark_background')
CYAN, GREEN, RED, MUTED = '#22d3ee', '#4ade80', '#f87171', '#a1a1aa'

---
## Ejercicio 0 — Antes de la ingeniería: ¿qué problema resolvemos?

*Sin código. Solo reflexión.*

**a)** En L5 colocaste un limit bid a $99,960 con el LOB de BTC. ¿Cómo sabrías si esa orden va a ejecutarse antes de que el precio llegue hasta allí?

**b)** Si el imbalance sube a 0.8, ¿qué esperarías que haga el precio? ¿Siempre? ¿A veces? ¿Cómo lo medirías objetivamente?

**c)** ¿Qué diferencia hay entre predecir bien *en los datos que ya tienes* vs predecir bien *en datos futuros que aún no existen*?

---
## ── NÚCLEO ──────────────────────────────────────────────────────

## Ejercicio 1 — Cargar datos y verificar features

Carga `../data/lob_features.csv`. Verifica que tiene 499 filas.  
Calcula `spread_mean` (media del spread) e `imbalance_std` (desviación estándar del imbalance).

In [ ]:
df = None           # ← carga el CSV aquí
spread_mean   = None
imbalance_std = None

In [ ]:
# Validador 1
assert df is not None, "df no está definido — carga el CSV primero"
assert len(df) == 499, f"Se esperaban 499 filas, hay {len(df)} — ¿cargaste lob_features.csv?"
assert 'imbalance' in df.columns, "Columna 'imbalance' no encontrada en el DataFrame"
assert 'spread' in df.columns, "Columna 'spread' no encontrada en el DataFrame"
assert spread_mean is not None, "spread_mean no está definido"
assert imbalance_std is not None, "imbalance_std no está definido"
assert abs(spread_mean - 11.396794) < 1e-4, f"spread_mean incorrecto: {spread_mean:.6f} (esperado ~11.3968)"
assert abs(imbalance_std - 0.097870) < 1e-4, f"imbalance_std incorrecto: {imbalance_std:.6f} (esperado ~0.0979)"
print('✓ Ejercicio 1 correcto')

In [ ]:
# Solución guiada 1
df = pd.read_csv('../data/lob_features.csv')
spread_mean   = df['spread'].mean()
imbalance_std = df['imbalance'].std()
print(f'Filas: {len(df)}')
print(f'spread_mean: {spread_mean:.4f}')
print(f'imbalance_std: {imbalance_std:.4f}')

---
## Ejercicio 2 — Definir el target de dirección

El CSV ya tiene columnas `mid_next` y `direction`. Pero reconstruye `direction` tú mismo para entender el proceso:

1. `direction_manual = 1` si `mid_next > mid`, `0` en caso contrario
2. Cuenta cuántos 1s (`up_count`) y cuántos 0s (`down_count`) hay.

In [ ]:
df['direction_manual'] = None   # ← define el target aquí
up_count   = None
down_count = None

In [ ]:
# Validador 2
assert df['direction_manual'].notna().all(), "direction_manual tiene NaN — aplícalo a todas las filas"
assert set(df['direction_manual'].unique()).issubset({0, 1}), "direction_manual debe contener solo 0 y 1"
assert up_count is not None and down_count is not None, "up_count o down_count no están definidos"
assert up_count + down_count == 499, f"up_count + down_count debe ser 499, es {up_count + down_count}"
assert up_count == 246, f"up_count incorrecto: {up_count} (esperado 246)"
assert down_count == 253, f"down_count incorrecto: {down_count} (esperado 253)"
# Verifica que coincide con la columna original
assert (df['direction_manual'] == df['direction']).all(), "direction_manual no coincide con 'direction' del CSV"
print('✓ Ejercicio 2 correcto')
print(f'  Subidas: {up_count} ({up_count/499*100:.1f}%)  |  Bajadas: {down_count} ({down_count/499*100:.1f}%)')

In [ ]:
# Solución guiada 2
df['direction_manual'] = (df['mid_next'] > df['mid']).astype(int)
up_count   = int((df['direction_manual'] == 1).sum())
down_count = int((df['direction_manual'] == 0).sum())
print(f'UP: {up_count}, DOWN: {down_count}')

---
## Ejercicio 3 — Split temporal

Divide `df` en `train_df` (primeras 70% filas) y `test_df` (últimas 30%).  
Asigna `n_train` y `n_test` como enteros.  
**Importante:** no uses `sklearn.train_test_split` — usa slicing directo con `.iloc`.

In [ ]:
train_df = None
test_df  = None
n_train  = None
n_test   = None

In [ ]:
# Validador 3
assert train_df is not None and test_df is not None, "train_df o test_df no definidos"
assert n_train == 349, f"n_train debe ser 349, es {n_train}"
assert n_test  == 150, f"n_test debe ser 150, es {n_test}"
assert n_train + n_test == 499, "n_train + n_test debe ser 499"
assert train_df.index.max() < test_df.index.min(), \
    "El índice máximo de train debe ser menor al mínimo de test — ¿respetaste el orden temporal?"
print('✓ Ejercicio 3 correcto')
print(f'  Train: índices {train_df.index.min()}–{train_df.index.max()}')
print(f'  Test:  índices {test_df.index.min()}–{test_df.index.max()}')

In [ ]:
# Solución guiada 3
split_idx = int(len(df) * 0.7)   # 349
train_df = df.iloc[:split_idx].copy()
test_df  = df.iloc[split_idx:].copy()
n_train  = len(train_df)
n_test   = len(test_df)
print(f'Train: {n_train}  |  Test: {n_test}')

---
## Ejercicio 4 — Detectar leakage

Tres listas de features. Una de ellas contiene un feature leaky.

```python
features_A = ['imbalance', 'spread_pct', 'wmid']
features_B = ['imbalance', 'spread_pct', 'realized_spread']   # ← ?
features_C = ['spread_pct', 'depth_ratio', 'imbalance']
```

Analiza qué es `realized_spread` mirando las columnas del CSV.  
Asigna `leaky_feature_set = 'A'`, `'B'` o `'C'`.  
Luego verifica tu hipótesis entrenando un LR con cada lista y comparando accuracies.

In [ ]:
features_A = ['imbalance', 'spread_pct', 'wmid']
features_B = ['imbalance', 'spread_pct', 'realized_spread']
features_C = ['spread_pct', 'depth_ratio', 'imbalance']

leaky_feature_set = None   # ← 'A', 'B' o 'C'

In [ ]:
# Validador 4
assert leaky_feature_set in ('A', 'B', 'C'), "leaky_feature_set debe ser 'A', 'B' o 'C'"
assert leaky_feature_set == 'B', (
    f"Set '{leaky_feature_set}' no es el correcto. Pista: ¿qué significa realized_spread? "
    "¿Usa información del momento t o de t+1?"
)
print('✓ Ejercicio 4 correcto')

In [ ]:
# Solución guiada 4
# realized_spread = ask_price_1[t+1] - bid_price_1[t]
# ask_price_1[t+1] es el ask del SIGUIENTE snapshot — no existe en el momento t.
# Se construyó con .shift(-1), que desplaza el futuro al pasado.
leaky_feature_set = 'B'

# Verificación empírica
def acc_lr(train, test, feats):
    lr = LogisticRegression(random_state=42, max_iter=1000)
    lr.fit(train[feats], train['direction'])
    return (lr.predict(test[feats]) == test['direction'].values).mean()

# realized_spread tiene NaN si hay filas sin t+1 — filtramos
train_l = train_df.dropna(subset=['realized_spread'])
test_l  = test_df.dropna(subset=['realized_spread'])

for name, feats in [('A', features_A), ('C', features_C)]:
    print(f'  Set {name}: {acc_lr(train_df, test_df, feats):.1%}')
print(f'  Set B (leaky): {acc_lr(train_l, test_l, features_B):.1%}  ← salto artificialmente alto')

---
## Ejercicio 5 — Baseline: siempre UP

Implementa `baseline_always_up(y_test)` que devuelve la accuracy si siempre predices 1 (UP).  
Aplícalo al `test_df` y asigna el resultado a `accuracy_always_up`.

In [ ]:
def baseline_always_up(y_test):
    pass  # ← implementa aquí

accuracy_always_up = None

In [ ]:
# Validador 5
assert accuracy_always_up is not None, "accuracy_always_up no está definido"
assert isinstance(accuracy_always_up, float), f"Debe ser float, es {type(accuracy_always_up)}"
assert 0.4 < accuracy_always_up < 0.7, f"Valor fuera de rango: {accuracy_always_up:.4f}"
assert abs(accuracy_always_up - 0.506667) < 1e-4, \
    f"Incorrecto: {accuracy_always_up:.4f} (esperado ~0.5067)"
print('✓ Ejercicio 5 correcto')
print(f'  Predecir siempre UP → accuracy: {accuracy_always_up:.1%}')
print('  Este es tu suelo: cualquier modelo debe superar esto para ser útil.')

In [ ]:
# Solución guiada 5
def baseline_always_up(y_test):
    y_arr = np.array(y_test)
    preds = np.ones(len(y_arr), dtype=int)
    return float((preds == y_arr).mean())

accuracy_always_up = baseline_always_up(test_df['direction'])
print(f'accuracy_always_up = {accuracy_always_up:.4f}')

---
## ── SI VAMOS BIEN ────────────────────────────────────────────────

## Ejercicio 6 — Threshold classifier

Implementa `threshold_predict(df_test, col, threshold)` que devuelve un array de predicciones:  
- 1 (UP) si `df_test[col] > threshold`  
- 0 (DOWN) en caso contrario

Aplícalo con `col='imbalance'`, `threshold=0.6` y calcula `accuracy_threshold`.

In [ ]:
def threshold_predict(df_test, col, threshold):
    pass  # ← implementa aquí

accuracy_threshold = None

In [ ]:
# Validador 6
assert accuracy_threshold is not None, "accuracy_threshold no definido"
assert isinstance(accuracy_threshold, float), "Debe ser float"
assert abs(accuracy_threshold - 0.54) < 1e-4, \
    f"Incorrecto: {accuracy_threshold:.4f} (esperado 0.5400 con threshold=0.6)"
assert accuracy_threshold > accuracy_always_up, \
    "El threshold classifier debe superar a 'siempre UP' — revisa el threshold"
print('✓ Ejercicio 6 correcto')
print(f'  Threshold (imbalance > 0.6) → accuracy: {accuracy_threshold:.1%}')

In [ ]:
# Solución guiada 6
def threshold_predict(df_test, col, threshold):
    return (df_test[col] > threshold).astype(int).values

preds_thresh    = threshold_predict(test_df, 'imbalance', 0.6)
accuracy_threshold = float((preds_thresh == test_df['direction'].values).mean())
print(f'accuracy_threshold = {accuracy_threshold:.4f}')

---
## Ejercicio 7 — Comparar features

Usa `threshold_predict` con el umbral = mediana de cada feature para los tres features:  
`imbalance`, `spread_pct`, `depth_ratio`.

¿Cuál da mayor accuracy en el test set? Asigna `best_feature` con el nombre del feature ganador.

In [ ]:
feature_candidates = ['imbalance', 'spread_pct', 'depth_ratio']
best_feature = None   # ← nombre del mejor feature

In [ ]:
# Validador 7
assert best_feature in feature_candidates, \
    f"best_feature debe ser uno de {feature_candidates}, es '{best_feature}'"
assert best_feature == 'depth_ratio', (
    f"'{best_feature}' no es el mejor. Pista: depth_ratio usa los 10 niveles del libro "
    "y captura la presión total de compradores vs vendedores."
)
print('✓ Ejercicio 7 correcto')
print(f'  Mejor feature: {best_feature}')

In [ ]:
# Solución guiada 7
accs = {}
for feat in feature_candidates:
    median = test_df[feat].median()
    preds  = threshold_predict(test_df, feat, median)
    accs[feat] = float((preds == test_df['direction'].values).mean())
    print(f'  {feat:<15} (umbral mediana={median:.4f}) → accuracy: {accs[feat]:.4f}')

best_feature = max(accs, key=accs.get)
print(f'\nMejor feature: {best_feature} ({accs[best_feature]:.1%})')

---
## ── BONUS / CASA ─────────────────────────────────────────────────

## Ejercicio 8 — LogisticRegression baseline

Entrena un `LogisticRegression` con features `['imbalance', 'spread_pct']` en el train set.  
Evalúa en el test set. Asigna `lr_accuracy`, `lr_precision` y `lr_recall`.

In [ ]:
from sklearn.metrics import precision_score, recall_score

lr_accuracy  = None
lr_precision = None
lr_recall    = None

In [ ]:
# Validador 8
assert lr_accuracy is not None, "lr_accuracy no definido"
assert 0.3 < lr_accuracy < 0.7, f"lr_accuracy fuera de rango: {lr_accuracy:.4f}"
assert abs(lr_accuracy - 0.493333) < 1e-4, \
    f"lr_accuracy incorrecto: {lr_accuracy:.4f} (esperado ~0.4933)"
assert 0 <= lr_precision <= 1, "lr_precision debe estar entre 0 y 1"
assert 0 <= lr_recall    <= 1, "lr_recall debe estar entre 0 y 1"
print('✓ Ejercicio 8 correcto')
print(f'  LR accuracy: {lr_accuracy:.1%}  — ¡por debajo del baseline siempre UP!')
print(f'  Precision: {lr_precision:.3f}  |  Recall: {lr_recall:.3f}')
print('  Esto es normal con pocos datos y features simples. La señal es débil.')

In [ ]:
# Solución guiada 8
feats = ['imbalance', 'spread_pct']
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(train_df[feats], train_df['direction'])
preds    = lr.predict(test_df[feats])
y_test_  = test_df['direction'].values

lr_accuracy  = float((preds == y_test_).mean())
lr_precision = float(precision_score(y_test_, preds, zero_division=0))
lr_recall    = float(recall_score(y_test_, preds, zero_division=0))
print(f'Accuracy: {lr_accuracy:.4f}  |  Precision: {lr_precision:.4f}  |  Recall: {lr_recall:.4f}')

---
## Ejercicio 9 — Confusion matrix manual

Implementa `confusion_matrix_manual(y_true, y_pred)` sin usar sklearn.  
Debe devolver un diccionario con claves `'tp'`, `'fp'`, `'tn'`, `'fn'`.

In [ ]:
def confusion_matrix_manual(y_true, y_pred):
    pass  # ← implementa aquí

cm_manual = None   # ← aplica con las predicciones del LR del ejercicio 8

In [ ]:
# Validador 9
assert cm_manual is not None, "cm_manual no definido — aplica confusion_matrix_manual"
assert isinstance(cm_manual, dict), "cm_manual debe ser un diccionario"
assert set(cm_manual.keys()) == {'tp', 'fp', 'tn', 'fn'}, \
    f"Claves incorrectas: {set(cm_manual.keys())}"
y_t = test_df['direction'].values
assert cm_manual['tp'] + cm_manual['fn'] == int((y_t == 1).sum()), \
    "tp + fn debe ser igual al número de positivos reales"
assert cm_manual['tn'] + cm_manual['fp'] == int((y_t == 0).sum()), \
    "tn + fp debe ser igual al número de negativos reales"
print('✓ Ejercicio 9 correcto')
print(f'  TP={cm_manual["tp"]}  FP={cm_manual["fp"]}  TN={cm_manual["tn"]}  FN={cm_manual["fn"]}')

In [ ]:
# Solución guiada 9
def confusion_matrix_manual(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return {
        'tp': int(((y_pred == 1) & (y_true == 1)).sum()),
        'fp': int(((y_pred == 1) & (y_true == 0)).sum()),
        'tn': int(((y_pred == 0) & (y_true == 0)).sum()),
        'fn': int(((y_pred == 0) & (y_true == 1)).sum()),
    }

# Usamos las predicciones del ejercicio 8
cm_manual = confusion_matrix_manual(test_df['direction'], preds)
print(cm_manual)

---
## Ejercicio 10 — Reflexión crítica (casa)

*Sin código. Reflexión escrita.*

Entrenaste un LR con accuracy ~49% — por debajo del baseline siempre UP (50.7%).

Responde en el siguiente campo de texto:
1. ¿Usarías este modelo en producción para operar BTC? Da 3 razones en contra.
2. ¿Bajo qué 2 condiciones podría tener valor real?
3. En L7 vamos a intentar mejorar este resultado. ¿Qué cambiarías primero: los features, el modelo, o la cantidad de datos? ¿Por qué?

*(Escribe tu respuesta en la celda de abajo)*

*Tu respuesta aquí...*

**Respuesta modelo:**

**3 razones en contra:**
1. Accuracy del 49.3% está por debajo del azar corregido por clase (50.7%). El modelo destruye valor.
2. Con 349 datos de entrenamiento, el modelo tiene altísima varianza — los pesos LR no son estables.
3. Los costes de transacción (comisión + spread bid/ask) consumen la pequeña ventaja estadística aunque fuera real.

**2 condiciones para que tenga valor:**
1. Como componente de un ensemble más complejo, donde la señal débil se combina con otras señales ortogonales.
2. Si el mercado es menos eficiente (crypto de baja liquidez, renta fija corporativa ilíquida) y la señal se mantiene con más datos.

**¿Qué cambiar primero?** Los features. Con 499 puntos y 2 features, el modelo LR tiene pocas opciones. Antes de aumentar la complejidad del modelo, necesitamos features con más poder predictivo — lo cual es exactamente lo que explora L7.